In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/abhinav2807/retinal55/DRIVE CHASE STARE DATASET/CHASE/test/images/test_02_test.tif
/kaggle/input/datasets/abhinav2807/retinal55/DRIVE CHASE STARE DATASET/CHASE/test/images/test_04_test.tif
/kaggle/input/datasets/abhinav2807/retinal55/DRIVE CHASE STARE DATASET/CHASE/test/images/test_03_test.tif
/kaggle/input/datasets/abhinav2807/retinal55/DRIVE CHASE STARE DATASET/CHASE/test/images/test_08_test.tif
/kaggle/input/datasets/abhinav2807/retinal55/DRIVE CHASE STARE DATASET/CHASE/test/images/test_01_test.tif
/kaggle/input/datasets/abhinav2807/retinal55/DRIVE CHASE STARE DATASET/CHASE/test/images/test_06_test.tif
/kaggle/input/datasets/abhinav2807/retinal55/DRIVE CHASE STARE DATASET/CHASE/test/images/test_07_test.tif
/kaggle/input/datasets/abhinav2807/retinal55/DRIVE CHASE STARE DATASET/CHASE/test/images/test_05_test.tif
/kaggle/input/datasets/abhinav2807/retinal55/DRIVE CHASE STARE DATASET/CHASE/test/masks/test_04_manual1.tif
/kaggle/input/datasets/abhinav2807/retinal55

In [2]:
!pip install albumentations thop medpy scikit-image -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 3.2 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done


In [3]:
import os, cv2, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, roc_auc_score, matthews_corrcoef
from skimage.morphology import skeletonize
import matplotlib.pyplot as plt
from medpy.metric.binary import hd95
from thop import profile

# ================= SETTINGS =================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE_PATH = "/kaggle/input/datasets/abhinav2807/retinal55/DRIVE CHASE STARE DATASET/CHASE"
OUTPUT_DIR = "chase_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

EPOCHS = 100
BATCH_SIZE = 4
LR = 1e-3

# ================= UTILS =================
def apply_clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(l)
    return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)

def get_cldice(p, l):
    p, l = p.astype(bool), l.astype(bool)
    if p.sum() == 0 or l.sum() == 0: return 0
    sp, sl = skeletonize(p), skeletonize(l)
    tprec = (sp * l).sum() / (sp.sum() + 1e-7)
    tsens = (sl * p).sum() / (sl.sum() + 1e-7)
    return 2 * (tprec * tsens) / (tprec + tsens + 1e-7)

# ================= DATASET =================
class ChaseDataset(Dataset):
    def __init__(self, mode="train"):
        self.image_dir = f"{BASE_PATH}/{mode}/images"
        m1, m2 = f"{BASE_PATH}/{mode}/mask", f"{BASE_PATH}/{mode}/masks"
        self.mask_dir = m1 if os.path.exists(m1) else m2
        self.images = sorted(os.listdir(self.image_dir))
        self.masks = sorted(os.listdir(self.mask_dir))
        
        self.transform = A.Compose([
            A.Resize(512, 512),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.Normalize(),
            ToTensorV2()
        ])

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        img = cv2.imread(os.path.join(self.image_dir, self.images[idx]))
        img = apply_clahe(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        mask = cv2.imread(os.path.join(self.mask_dir, self.masks[idx]), 0)
        mask = (mask > 0).astype(np.uint8)
        
        aug = self.transform(image=img, mask=mask)
        return self.images[idx], aug['image'], aug['mask'].unsqueeze(0).float()

# ================= MODEL (U-Net) =================
class DoubleConv(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(i, o, 3, 1, 1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
            nn.Conv2d(o, o, 3, 1, 1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.ModuleList([DoubleConv(3, 64), DoubleConv(64, 128), DoubleConv(128, 256), DoubleConv(256, 512)])
        self.pool = nn.MaxPool2d(2)
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.dec = nn.ModuleList([DoubleConv(512+256, 256), DoubleConv(256+128, 128), DoubleConv(128+64, 64)])
        self.final = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        skips = []
        for layer in self.enc[:-1]:
            x = layer(x)
            skips.append(x)
            x = self.pool(x)
        x = self.enc[-1](x)
        for i, layer in enumerate(self.dec):
            x = self.up(x)
            x = layer(torch.cat([x, skips[-(i+1)]], dim=1))
        return self.final(x)

# ================= LOSS & METRICS =================
class ComboLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
    def forward(self, p, t):
        bce = self.bce(p, t)
        p = torch.sigmoid(p)
        inter = (p * t).sum()
        dice = 1 - (2 * inter + 1e-7) / (p.sum() + t.sum() + 1e-7)
        return bce + dice

def calc_metrics(pred, gt):
    pb = (pred > 0.5).astype(np.uint8)
    gt = gt.astype(np.uint8)
    tn, fp, fn, tp = confusion_matrix(gt.flatten(), pb.flatten(), labels=[0,1]).ravel()
    
    # Calculate key metrics
    metrics = {
        "ACC": (tp + tn) / (tp + tn + fp + fn + 1e-7),
        "Dice": (2 * tp) / (2 * tp + fp + fn + 1e-7),
        "IoU": tp / (tp + fp + fn + 1e-7),
        "Sen": tp / (tp + fn + 1e-7),
        "Spe": tn / (tn + fp + 1e-7),
        "AUC": roc_auc_score(gt.flatten(), pred.flatten()),
        "MCC": matthews_corrcoef(gt.flatten(), pb.flatten()),
        "clDice": get_cldice(pb, gt),
        "HD95": hd95(pb, gt) if pb.sum() > 0 else 100.0
    }
    return metrics

# ================= TRAINING =================
model = UNet().to(device)
train_loader = DataLoader(ChaseDataset("train"), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(ChaseDataset("test"), batch_size=1)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = ComboLoss()
scaler = torch.cuda.amp.GradScaler()

best_dice = 0
for epoch in range(EPOCHS):
    model.train()
    for _, img, mask in train_loader:
        img, mask = img.to(device), mask.to(device)
        with torch.cuda.amp.autocast():
            loss = criterion(model(img), mask)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    scheduler.step()
    
    # Simple Val
    model.eval()
    dices = []
    with torch.no_grad():
        for _, img, mask in test_loader:
            p = torch.sigmoid(model(img.to(device)))
            inter = (p * mask.to(device)).sum()
            dices.append((2 * inter / (p.sum() + mask.sum() + 1e-7)).item())
    
    avg_dice = np.mean(dices)
    if avg_dice > best_dice:
        best_dice = avg_dice
        torch.save(model.state_dict(), "best_chase.pth")
    print(f"Epoch {epoch+1} | Dice: {avg_dice:.4f}")

# ================= FINAL EVALUATION =================
model.load_state_dict(torch.load("best_chase.pth"))
model.eval()

results = []
with torch.no_grad():
    for name, img, mask in tqdm(test_loader):
        start = time.time()
        pred = torch.sigmoid(model(img.to(device)))
        inf_time = time.time() - start
        
        pred_np, mask_np = pred.cpu().numpy().squeeze(), mask.numpy().squeeze()
        m = calc_metrics(pred_np, mask_np)
        results.append(list(m.values()) + [inf_time])
        
        # Save output comparison
        plt.figure(figsize=(12, 4))
        plt.subplot(131); plt.imshow(apply_clahe(cv2.imread(os.path.join(BASE_PATH,"test/images",name[0])))); plt.title("Original")
        plt.subplot(132); plt.imshow(mask_np, cmap='gray'); plt.title("GT")
        plt.subplot(133); plt.imshow(pred_np > 0.5, cmap='gray'); plt.title(f"Dice: {m['Dice']:.3f}")
        plt.savefig(os.path.join(OUTPUT_DIR, f"res_{name[0]}.png"))
        plt.close()

# Summary
avg_metrics = np.mean(results, axis=0)
headers = list(m.keys()) + ["InferenceTime"]
print("\n" + "="*80)
for h, v in zip(headers, avg_metrics):
    print(f"{h:<15}: {v:.4f}")

/tmp/ipykernel_56/3375961729.py:139: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
[ WARN:0@14.511] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20624 (0x5090) encountered
[ WARN:0@14.511] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20625 (0x5091) encountered
[ WARN:0@14.785] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20752 (0x5110) encountered
[ WARN:0@14.785] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20753 (0x5111) encountered
[ WARN:0@14.785] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20754 (0x5112) encountered
[ WARN:0@14.862] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20624 (0x5090) encountered
[ WARN:0@14.862] global grfmt_tiff.cpp:122 TIFF

Epoch 1 | Dice: 0.1085


[ WARN:0@22.467] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20752 (0x5110) encountered
[ WARN:0@22.467] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20753 (0x5111) encountered
[ WARN:0@22.467] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20754 (0x5112) encountered
[ WARN:0@22.481] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20624 (0x5090) encountered
[ WARN:0@22.481] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20625 (0x5091) encountered
[ WARN:0@22.519] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20752 (0x5110) encountered
[ WARN:0@22.519] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20753 (0x5111) encountered
[ WARN:0@22.519] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20754 (0x5112) encountered


Epoch 2 | Dice: 0.1100


[ WARN:0@26.241] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20624 (0x5090) encountered
[ WARN:0@26.241] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20625 (0x5091) encountered
[ WARN:0@26.277] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 769 (0x301) encountered
[ WARN:0@26.277] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 771 (0x303) encountered
[ WARN:0@26.277] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20752 (0x5110) encountered
[ WARN:0@26.277] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20753 (0x5111) encountered
[ WARN:0@26.277] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20754 (0x5112) encountered
[ WARN:0@26.291] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20624 (0x5090) encountered
[ WARN

...

Epoch 99 | Dice: 0.7234


[ WARN:0@433.327] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20624 (0x5090) encountered
[ WARN:0@433.327] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20625 (0x5091) encountered
[ WARN:0@433.367] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20752 (0x5110) encountered
[ WARN:0@433.367] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20753 (0x5111) encountered
[ WARN:0@433.367] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20754 (0x5112) encountered
[ WARN:0@433.383] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20624 (0x5090) encountered
[ WARN:0@433.383] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20625 (0x5091) encountered
[ WARN:0@433.420] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20752 (0x5110) enco

Epoch 100 | Dice: 0.7238


  0%|          | 0/8 [00:00<?, ?it/s][ WARN:0@437.655] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20624 (0x5090) encountered
[ WARN:0@437.655] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20625 (0x5091) encountered
[ WARN:0@437.688] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20752 (0x5110) encountered
[ WARN:0@437.688] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20753 (0x5111) encountered
[ WARN:0@437.688] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20754 (0x5112) encountered
[ WARN:0@437.988] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20624 (0x5090) encountered
[ WARN:0@437.988] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 20625 (0x5091) encountered
 12%|█▎        | 1/8 [00:00<00:05,  1.17it/s][ WARN:0@438.511] global grfmt_tiff.


ACC            : 0.9684
Dice           : 0.7601
IoU            : 0.6134
Sen            : 0.7946
Spe            : 0.9801
AUC            : 0.9818
MCC            : 0.7448
clDice         : 0.7762
HD95           : 7.1248
InferenceTime  : 0.0038
